# 🧠 Agent Metacognition with Microsoft Agent Framework (C#)

## Overview

This notebook demonstrates **metacognition** in AI agents - the ability for agents to be aware of and reason about their own thought processes and decision-making. We'll build a travel booking agent that:

- **Learns from interactions**: Remembers user preferences from previous conversations
- **Self-reflects**: Evaluates its own suggestions and adjusts based on feedback
- **Maintains context**: Tracks conversation history to provide consistent recommendations
- **Reasons about time**: Validates its own suggestions for feasibility

## What is Metacognition?

**Metacognition** is "thinking about thinking" - an agent's ability to:
1. Monitor its own reasoning process
2. Learn from past interactions
3. Adjust behavior based on feedback
4. Self-validate suggestions before presenting them

## Key Concepts Demonstrated

- 🧠 **Preference Learning**: Agent remembers and applies user preferences across destinations
- 🔄 **Self-Reflection**: Agent validates its own suggestions for reasonableness
- 📊 **Context Tracking**: Maintains customer preferences throughout the conversation
- 🎯 **Adaptive Behavior**: Adjusts recommendations based on user feedback

## Architecture

```
User Input → Agent (with Metacognition)
                ↓
        [Self-Reflection Layer]
                ↓
    Check Customer Preferences
                ↓
        Call Functions (Tools)
                ↓
        Validate Suggestions
                ↓
        Update Preferences
                ↓
        Provide Response
```

In [ ]:
Console.WriteLine("Environment Variables:");

In [ ]:
// 📦 Install Required NuGet Packages
#r "nuget: Microsoft.AgentFramework.Core"
#r "nuget: Azure.Identity"

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;

// Azure authentication
using Azure.Identity;

// Agent Framework
using Microsoft.AgentFramework;
using Microsoft.AgentFramework.Azure;

## Define Tool Functions

These functions provide the agent with capabilities to retrieve destinations and flight times.

In [ ]:
// Tool Functions for the Agent
// These functions will be available to the agent as tools

string GetDestinations()
{
    return @"
    Barcelona, Spain
    Paris, France
    Berlin, Germany
    Tokyo, Japan
    New York, USA
    ";
}

string GetFlightTimes(string destination)
{
    var flightTimes = new Dictionary<string, string[]>
    {
        { "Barcelona", new[] { "08:30 AM", "02:15 PM", "10:45 PM" } },
        { "Paris", new[] { "06:45 AM", "12:30 PM", "07:15 PM" } },
        { "Berlin", new[] { "07:20 AM", "01:45 PM", "09:30 PM" } },
        { "Tokyo", new[] { "11:00 AM", "05:30 PM", "11:55 PM" } },
        { "New York", new[] { "05:15 AM", "03:00 PM", "08:45 PM" } }
    };

    // Extract just the city name from input that might contain country
    var city = destination.Split(',')[0].Trim();

    if (flightTimes.ContainsKey(city))
    {
        var times = string.Join(", ", flightTimes[city]);
        return $"Flight times for {city}: {times}";
    }
    else
    {
        return $"No flight information available for {city}.";
    }
}

Console.WriteLine("✅ Tool functions defined");

## Load Environment Configuration

In [ ]:
var projectEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_PROJECT_ENDPOINT");
var modelDeploymentName = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL");

Console.WriteLine($"Project Endpoint: {projectEndpoint}");
Console.WriteLine($"Model: {modelDeploymentName}");

## Create the Travel Agent with Metacognition

This agent demonstrates metacognitive capabilities through:
- Tracking customer preferences
- Learning from past interactions
- Self-validating suggestions
- Adapting behavior based on feedback

In [ ]:
const string AGENT_NAME = "TravelAgent";
const string AGENT_INSTRUCTIONS = @"
You are Flight Booking Agent that provides information about available flights and gives travel activity suggestions when asked.
Travel activity suggestions should be specific to customer, location and amount of time at location.

You have access to the following tools to help users plan their trips:
1. GetDestinations: Returns a list of available vacation destinations that users can choose from.
2. GetFlightTimes: Provides available flight times for specific destinations.


Your process for assisting users:
- When users first inquire about flight booking with no prior history, ask for their preferred flight time ONCE.
- MAINTAIN a customer_preferences object throughout the conversation to track preferred flight times.
- When a user books a flight to any destination, RECORD their chosen flight time in the customer_preferences object.
- For ALL subsequent flight inquiries to ANY destination, AUTOMATICALLY apply their existing preferred flight time without asking.
- NEVER ask about time preferences again after they've been established for any destination.
- When suggesting flights for a new destination, explicitly say: ""Based on your previous preference for [time] flights, I recommend...""
- Only after showing options matching their preferred time, ask if they want to see alternative times.
- After each booking, UPDATE the customer_preferences object with any new information.
- ALWAYS mention which specific preference you used when making a suggestion.

Guidelines:
- Use the exact destination names when using tools (Barcelona, Paris, Berlin, Tokyo, New York)
- Respond in a helpful and enthusiastic manner about travel possibilities
- Always seek feedback to ensure your suggestions meet the user's expectations
- Acknowledge when a request falls outside your capabilities
- For better formatting, always display flight times in a list format
- When giving any timed suggestions, reflect if the time frames are reasonable. Respond again if not.

Your goal is to help users explore vacation options efficiently and make informed travel decisions by understanding their preferences and providing tailored recommendations.
";

// Create Azure AI Agent Client
var credential = new AzureCliCredential();

var agentChatClient = new AzureAIAgentClient(
    credential: credential,
    modelDeploymentName: modelDeploymentName,
    projectEndpoint: new Uri(projectEndpoint)
);

// Create the chat agent with tools
var agent = new ChatAgent(
    name: AGENT_NAME,
    chatClient: agentChatClient,
    instructions: AGENT_INSTRUCTIONS,
    tools: new Delegate[] { GetDestinations, GetFlightTimes }
);

Console.WriteLine($"✅ Agent '{AGENT_NAME}' created with metacognition capabilities");

## Helper Functions for Display

In [ ]:
void DisplayUserMessage(string message)
{
    Console.WriteLine();
    Console.ForegroundColor = ConsoleColor.Cyan;
    Console.WriteLine($"👤 User:");
    Console.ResetColor();
    Console.WriteLine($"   {message}");
    Console.WriteLine();
}

void DisplayAgentResponse(string agentName, string response)
{
    Console.ForegroundColor = ConsoleColor.Green;
    Console.WriteLine($"🤖 {agentName}:");
    Console.ResetColor();
    Console.WriteLine($"   {response}");
    Console.WriteLine(new string('-', 80));
}

void DisplayFunctionCall(string functionName, string arguments)
{
    Console.ForegroundColor = ConsoleColor.Yellow;
    Console.WriteLine($"   🔧 Function Call: {functionName}({arguments})");
    Console.ResetColor();
}

Console.WriteLine("✅ Display helpers defined");

## Demonstration: Agent Learning User Preferences

Watch how the agent:
1. Asks for preferences initially
2. Learns from user feedback
3. Remembers preferences for future interactions
4. Self-validates time-based suggestions

In [ ]:
var userInputs = new List<string>
{
    "Book me a flight to Barcelona",
    "I prefer a later flight",
    "That is too late, choose the earliest flight",
    "I want to leave the same day, give me some suggestions of things to do in Barcelona during my layover if I take the last flight out",
    "I am stressed this wont be enough time"
};

// Create a thread to maintain conversation context
var thread = agent.GetNewThread();

foreach (var userInput in userInputs)
{
    DisplayUserMessage(userInput);
    
    var response = await agent.RunAsync(userInput, thread: thread);
    
    // Get the last message from the response
    var lastMessage = response.Messages[response.Messages.Count - 1];
    var textContent = lastMessage.Contents[0].Text;
    
    DisplayAgentResponse(AGENT_NAME, textContent);
}

Console.WriteLine("\n✅ First conversation sequence completed");

## Demonstration: Applying Learned Preferences to New Destination

Now watch how the agent applies previously learned preferences when booking a flight to a different destination. This demonstrates **metacognition** - the agent remembers and applies learned behavior.

In [ ]:
// Continue the conversation with a new destination
// Using the same thread maintains conversation context
var continuedInputs = new List<string>
{
    "Book me a flight to Paris"
};

foreach (var userInput in continuedInputs)
{
    DisplayUserMessage(userInput);
    
    var response = await agent.RunAsync(userInput, thread: thread);
    
    // Get the last message from the response
    var lastMessage = response.Messages[response.Messages.Count - 1];
    var textContent = lastMessage.Contents[0].Text;
    
    DisplayAgentResponse(AGENT_NAME, textContent);
}

Console.WriteLine("\n✅ Continued conversation completed");
Console.WriteLine("\n📊 Notice how the agent remembered the user's preference for early morning flights!");

## Summary: Metacognition in Action

This notebook demonstrated several key aspects of agent metacognition:

### 🧠 What We Observed:

1. **Preference Learning**
   - Agent asked about preferences initially
   - Remembered the user's preference for early morning flights
   - Applied this preference to a new destination (Paris) without asking again

2. **Self-Reflection**
   - Agent validated time-based suggestions
   - Adjusted recommendations when user expressed concerns
   - Acknowledged limitations when appropriate

3. **Context Awareness**
   - Maintained conversation history across multiple interactions
   - Referenced previous decisions when making new suggestions
   - Explicitly stated which preferences were being applied

4. **Adaptive Behavior**
   - Changed recommendations based on user feedback
   - Learned from corrections ("too late" → "earliest flight")
   - Provided contextual suggestions (layover activities)

### 🎯 Key Takeaways:

- **Metacognition enables personalization** at scale across different contexts
- **Agents can learn** from user feedback and apply it to future interactions
- **Self-reflection** helps agents provide more reasonable and validated suggestions
- **Context tracking** is essential for maintaining coherent, personalized conversations

### 🚀 Next Steps:

To enhance metacognition further:
- Add explicit preference storage and retrieval
- Implement confidence scoring for suggestions
- Add reasoning traces to show agent's thought process
- Include multi-turn planning capabilities
- Integrate with external memory systems for long-term learning